# Notebook 03 — Final 500-Patient 3D Patch Extraction

This notebook performs **patient-by-patient 3D patch extraction** for the 500 labelled RibFrac patients.

### Labelled cohort
- Training Part 1: 300 patients
- Training Part 2: 120 patients
- Labelled validation: 80 patients
- Total: **500 patients**

### Patient-level split
- Training: 350 patients
- Validation: 75 patients
- Internal held-out test: 75 patients

The split is established **before patch extraction** and is reproduced with the same fixed seed (`42`) and sorted 500-patient registry used by Notebook 02.

> The `test` split here is an **internal held-out labelled test set**, not the official RibFrac challenge test set.

### Patch policy
- Patch size: `(64, 64, 64)`
- Stride: `(32, 32, 32)`
- Segmentation target: `0 = background`, `1 = valid fracture`
- Classification target: `0 = background`, `1 = displaced`, `2 = non-displaced`, `3 = buckle`, `4 = segmental`
- NIfTI annotation IDs are mapped to fracture classes using patient metadata.
- `label_code = -1` is ambiguous and is **excluded**, never treated as background.
- Positive patches require at least one valid fracture voxel.
- Negative patches contain no valid fracture voxels.
- Mixed-class ties are excluded because the current classifier is single-label softmax.
- Negative sampling is capped at a **1:1 negative-to-positive ratio**.
- No patch-level train/validation/test split is performed.

The notebook uses the same validated on-demand Zenodo retrieval and preprocessing strategy as Notebook 02, so the full 1-mm volume set is never permanently materialized.

In [ ]:
# Cell 1 — Imports and reproducibility

import gc
import hashlib
import io
import json
import random
import struct
import tempfile
import time
import zipfile
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import nibabel as nib
from sklearn.model_selection import train_test_split
from scipy.ndimage import zoom

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Libraries imported successfully.")
print("Random seed:", SEED)

In [ ]:
# Cell 2 — Source configuration and output paths

SOURCES = {
    "train_part1": {
        "record": "3893508",
        "expected_patients": 300,
        "info_filename": "ribfrac-train-info-1.csv",
        "image_zip": "ribfrac-train-images-1.zip",
        "label_zip": "ribfrac-train-labels-1.zip",
        "image_url": "https://zenodo.org/records/3893508/files/ribfrac-train-images-1.zip",
        "label_url": "https://zenodo.org/records/3893508/files/ribfrac-train-labels-1.zip",
        "info_url": "https://zenodo.org/records/3893508/files/ribfrac-train-info-1.csv",
    },
    "train_part2": {
        "record": "3893498",
        "expected_patients": 120,
        "info_filename": "ribfrac-train-info-2.csv",
        "image_zip": "ribfrac-train-images-2.zip",
        "label_zip": "ribfrac-train-labels-2.zip",
        "image_url": "https://zenodo.org/records/3893498/files/ribfrac-train-images-2.zip",
        "label_url": "https://zenodo.org/records/3893498/files/ribfrac-train-labels-2.zip",
        "info_url": "https://zenodo.org/records/3893498/files/ribfrac-train-info-2.csv",
    },
    "validation": {
        "record": "3893496",
        "expected_patients": 80,
        "info_filename": "ribfrac-val-info.csv",
        "image_zip": "ribfrac-val-images.zip",
        "label_zip": "ribfrac-val-labels.zip",
        "image_url": "https://zenodo.org/records/3893496/files/ribfrac-val-images.zip",
        "label_url": "https://zenodo.org/records/3893496/files/ribfrac-val-labels.zip",
        "info_url": "https://zenodo.org/records/3893496/files/ribfrac-val-info.csv",
    },
}

PATCH_ROOT = Path("/kaggle/working/ribfrac-patch-dataset-final")
WORK_ROOT = Path("/kaggle/working/ribfrac_patch_extraction_work")
CACHE_ROOT = WORK_ROOT / "cache"
METADATA_ROOT = WORK_ROOT / "metadata"

for path in [PATCH_ROOT, CACHE_ROOT, METADATA_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("Patch output:", PATCH_ROOT)
print("Working root:", WORK_ROOT)

In [ ]:
# Cell 3 — Patch and target configuration

PATCH_SIZE = (64, 64, 64)
STRIDE = (32, 32, 32)

NEGATIVE_TO_POSITIVE_RATIO = 1.0
MIN_FRACTURE_VOXELS = 1
AMBIGUOUS_LABEL_CODE = -1

CLASS_NAMES = {
    0: "background",
    1: "displaced",
    2: "non-displaced",
    3: "buckle",
    4: "segmental",
}

LABEL_CODE_TO_CLASS = {
    1: 1,
    2: 2,
    3: 3,
    4: 4,
}

TARGET_SPACING = np.array([1.0, 1.0, 1.0], dtype=np.float32)
HU_MIN = -1000.0
HU_MAX = 400.0

print("Patch size:", PATCH_SIZE)
print("Stride:", STRIDE)
print("Negative/positive ratio:", NEGATIVE_TO_POSITIVE_RATIO)
print("Classification classes:", CLASS_NAMES)
print("HU normalization:", HU_MIN, "to", HU_MAX)

## Data-source strategy

The large image ZIP files are **not downloaded in full**. Their ZIP central directories are inspected remotely, and only the requested patient's compressed image member is downloaded using HTTP byte ranges.

The small label ZIP files and metadata CSV files are cached locally.

This reproduces the successful Notebook 02 on-demand preprocessing strategy.

In [ ]:
# Cell 4 — HTTP and small-file helpers

def http_get_bytes(url, start=None, end=None, timeout=120):
    headers = {
        "User-Agent": "Mozilla/5.0 RibFrac research preprocessing"
    }

    if start is not None and end is not None:
        headers["Range"] = f"bytes={start}-{end}"

    request = Request(url, headers=headers)

    with urlopen(request, timeout=timeout) as response:
        data = response.read()
        content_range = response.headers.get("Content-Range")

    return data, content_range


def get_remote_size(url):
    request = Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 RibFrac research preprocessing"},
        method="HEAD",
    )

    try:
        with urlopen(request, timeout=60) as response:
            size = response.headers.get("Content-Length")
            if size:
                return int(size)
    except Exception:
        pass

    probe, content_range = http_get_bytes(url, 0, 0)

    if content_range and "/" in content_range:
        return int(content_range.split("/")[-1])

    raise RuntimeError(f"Could not determine remote size: {url}")


def cache_small_file(url, destination):
    destination = Path(destination)

    if destination.exists() and destination.stat().st_size > 0:
        return destination

    print("Downloading small file:", destination.name)
    data, _ = http_get_bytes(url)

    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(data)

    print(
        f"Cached {destination.name}: "
        f"{destination.stat().st_size / (1024**2):.2f} MB"
    )

    return destination


print("HTTP helpers ready.")

In [ ]:
# Cell 5 — ZIP64 remote index helpers

def parse_zip64_extra(extra):
    pos = 0
    values = []

    while pos + 4 <= len(extra):
        header_id, size = struct.unpack(
            "<HH",
            extra[pos:pos + 4],
        )

        data = extra[pos + 4:pos + 4 + size]
        pos += 4 + size

        if header_id == 0x0001:
            values.append(data)

    return values


def read_zip64_central_directory(url):
    size = get_remote_size(url)

    tail_size = min(size, 1024 * 1024)
    tail, _ = http_get_bytes(
        url,
        size - tail_size,
        size - 1,
    )

    eocd_sig = b"PK"
    eocd_pos = tail.rfind(eocd_sig)

    if eocd_pos < 0:
        raise RuntimeError(
            "ZIP End Of Central Directory not found."
        )

    eocd = tail[eocd_pos:eocd_pos + 22]

    (
        _,
        _disk,
        _cd_disk,
        _entries_disk,
        entries_total,
        cd_size32,
        cd_offset32,
        _comment_len,
    ) = struct.unpack(
        "<4sHHHHIIH",
        eocd,
    )

    if (
        cd_size32 != 0xFFFFFFFF
        and cd_offset32 != 0xFFFFFFFF
    ):
        cd_size = cd_size32
        cd_offset = cd_offset32

    else:
        locator_sig = b"PK"
        locator_pos = tail.rfind(
            locator_sig,
            0,
            eocd_pos,
        )

        if locator_pos < 0:
            raise RuntimeError(
                "ZIP64 EOCD locator not found."
            )

        (
            _,
            _,
            zip64_eocd_offset,
            _,
        ) = struct.unpack(
            "<4sIQI",
            tail[locator_pos:locator_pos + 20],
        )

        z64, _ = http_get_bytes(
            url,
            zip64_eocd_offset,
            min(
                size - 1,
                zip64_eocd_offset + 1024 * 1024 - 1,
            ),
        )

        if z64[:4] != b"PK":
            raise RuntimeError(
                "ZIP64 EOCD record not found."
            )

        entries_total = struct.unpack_from(
            "<Q",
            z64,
            32,
        )[0]

        cd_size = struct.unpack_from(
            "<Q",
            z64,
            40,
        )[0]

        cd_offset = struct.unpack_from(
            "<Q",
            z64,
            48,
        )[0]

    cd_bytes, _ = http_get_bytes(
        url,
        cd_offset,
        cd_offset + cd_size - 1,
    )

    entries = {}
    pos = 0

    for _ in range(entries_total):

        if cd_bytes[pos:pos + 4] != b"PK":
            raise RuntimeError(
                "Invalid central-directory signature."
            )

        fixed = cd_bytes[pos:pos + 46]

        fields = struct.unpack(
            "<4s6H3I5H2I",
            fixed,
        )

        (
            _signature,
            _version_made,
            _version_needed,
            _flags,
            compression,
            _mod_time,
            _mod_date,
            _crc32,
            comp_size32,
            uncomp_size32,
            name_len,
            extra_len,
            comment_len,
            _disk_start,
            _int_attr,
            _ext_attr,
            local_offset32,
        ) = fields

        name_start = pos + 46

        name = cd_bytes[
            name_start:name_start + name_len
        ].decode(
            "utf-8",
            errors="replace",
        )

        extra_start = name_start + name_len

        extra = cd_bytes[
            extra_start:extra_start + extra_len
        ]

        comp_size = comp_size32
        uncomp_size = uncomp_size32
        local_offset = local_offset32

        if (
            comp_size32 == 0xFFFFFFFF
            or uncomp_size32 == 0xFFFFFFFF
            or local_offset32 == 0xFFFFFFFF
        ):
            chunks = parse_zip64_extra(extra)

            if not chunks:
                raise RuntimeError(
                    f"ZIP64 extra field missing for {name}"
                )

            data = chunks[0]
            zpos = 0

            if uncomp_size32 == 0xFFFFFFFF:
                uncomp_size = struct.unpack_from(
                    "<Q",
                    data,
                    zpos,
                )[0]
                zpos += 8

            if comp_size32 == 0xFFFFFFFF:
                comp_size = struct.unpack_from(
                    "<Q",
                    data,
                    zpos,
                )[0]
                zpos += 8

            if local_offset32 == 0xFFFFFFFF:
                local_offset = struct.unpack_from(
                    "<Q",
                    data,
                    zpos,
                )[0]

        entries[name] = {
            "compression": compression,
            "compressed_size": int(comp_size),
            "uncompressed_size": int(uncomp_size),
            "local_header_offset": int(local_offset),
        }

        pos = extra_start + extra_len + comment_len

    return size, entries


ZIP_INDEX_CACHE = {}

print("ZIP64 helper ready.")

In [ ]:
# Cell 6 — Patient ZIP-member retrieval

def normalize_member_name(name):
    return name.lstrip("/").replace("\\", "/")


def find_patient_member(entries, patient_id, kind):
    patient_id = str(patient_id)
    candidates = []

    for name in entries:
        base = Path(
            normalize_member_name(name)
        ).name

        if patient_id not in base:
            continue

        lower = base.lower()

        if kind == "image" and "image" in lower:
            candidates.append(name)

        elif kind == "label" and "label" in lower:
            candidates.append(name)

    if not candidates:
        raise FileNotFoundError(
            f"No {kind} ZIP member found for "
            f"patient {patient_id}."
        )

    exact_name = (
        f"{patient_id}-"
        f"{kind}.nii.gz"
    )

    exact = [
        name for name in candidates
        if Path(name).name == exact_name
    ]

    if len(exact) == 1:
        return exact[0]

    return sorted(candidates)[0]


def get_source_zip_index(source_name):
    if source_name in ZIP_INDEX_CACHE:
        return ZIP_INDEX_CACHE[source_name]

    cfg = SOURCES[source_name]

    print(
        f"Inspecting remote image ZIP: "
        f"{cfg['image_zip']}"
    )

    size, entries = read_zip64_central_directory(
        cfg["image_url"]
    )

    ZIP_INDEX_CACHE[source_name] = {
        "size": size,
        "entries": entries,
    }

    print(
        f"Remote ZIP indexed: {len(entries)} members, "
        f"{size / (1024**3):.2f} GB"
    )

    return ZIP_INDEX_CACHE[source_name]


def get_zip_member_bytes(
    url,
    entry,
    chunk_size=1024 * 1024,
):
    offset = entry["local_header_offset"]

    header, _ = http_get_bytes(
        url,
        offset,
        offset + 30 - 1,
    )

    if header[:4] != b"PK":
        raise RuntimeError(
            "Invalid local ZIP header."
        )

    (
        _signature,
        _version,
        _flags,
        compression,
        _mod_time,
        _mod_date,
        _crc32,
        _comp_size32,
        _uncomp_size32,
        name_len,
        extra_len,
    ) = struct.unpack(
        "<4s5H3I2H",
        header,
    )

    data_start = (
        offset
        + 30
        + name_len
        + extra_len
    )

    comp_size = entry["compressed_size"]
    data_end = data_start + comp_size - 1

    chunks = []

    for start in range(
        data_start,
        data_end + 1,
        chunk_size,
    ):
        end = min(
            start + chunk_size - 1,
            data_end,
        )

        chunk, _ = http_get_bytes(
            url,
            start,
            end,
        )

        chunks.append(chunk)

    compressed = b"".join(chunks)

    if compression == 0:
        return compressed

    if compression == 8:
        import zlib
        return zlib.decompress(
            compressed,
            -15,
        )

    raise RuntimeError(
        f"Unsupported ZIP compression method: "
        f"{compression}"
    )


print("Patient ZIP-member retrieval ready.")

In [ ]:
# Cell 8 — Reuse the exact Notebook 02 350/75/75 split procedure

# Notebook 02 used:
#   all_patients = np.array(sorted(combined_patient_ids))
#   train_test_split(..., test_size=150, random_state=42)
#   train_test_split(..., test_size=75, random_state=42)
#
# We reproduce that procedure here rather than creating a new split.

all_patients_array = np.array(
    sorted(all_patients)
)

train_patients, temp_patients = train_test_split(
    all_patients_array,
    test_size=150,
    random_state=SEED,
    shuffle=True,
)

validation_patients, test_patients = train_test_split(
    temp_patients,
    test_size=75,
    random_state=SEED,
    shuffle=True,
)

train_patients = sorted(
    train_patients.tolist()
)

validation_patients = sorted(
    validation_patients.tolist()
)

test_patients = sorted(
    test_patients.tolist()
)

train_set = set(train_patients)
validation_set = set(validation_patients)
test_set = set(test_patients)

assert len(train_set) == 350
assert len(validation_set) == 75
assert len(test_set) == 75

assert train_set.isdisjoint(validation_set)
assert train_set.isdisjoint(test_set)
assert validation_set.isdisjoint(test_set)

patient_registry = pd.DataFrame(
    [
        {
            "public_id": pid,
            "split": "train",
            "source": next(
                source
                for source, ids
                in source_patient_sets.items()
                if pid in ids
            ),
        }
        for pid in train_patients
    ]
    + [
        {
            "public_id": pid,
            "split": "validation",
            "source": next(
                source
                for source, ids
                in source_patient_sets.items()
                if pid in ids
            ),
        }
        for pid in validation_patients
    ]
    + [
        {
            "public_id": pid,
            "split": "test",
            "source": next(
                source
                for source, ids
                in source_patient_sets.items()
                if pid in ids
            ),
        }
        for pid in test_patients
    ]
)

patient_registry.to_csv(
    PATCH_ROOT / "patient_split_500.csv",
    index=False,
)

print(
    patient_registry.groupby(
        ["source", "split"]
    ).size()
)

print()
print("Train:", len(train_patients))
print("Validation:", len(validation_patients))
print("Internal test:", len(test_patients))
print("Patient-level split integrity: PASS")


In [ ]:
# Cell 8 — Reproduce the final 350/75/75 patient split

# This reproduces Notebook 02's patient-level split:
# sorted 500 patient IDs, seed 42, 70/15/15.

rng = random.Random(SEED)

shuffled_patients = list(all_patients)
rng.shuffle(shuffled_patients)

train_patients = sorted(
    shuffled_patients[:350]
)

validation_patients = sorted(
    shuffled_patients[350:425]
)

test_patients = sorted(
    shuffled_patients[425:500]
)

train_set = set(train_patients)
validation_set = set(validation_patients)
test_set = set(test_patients)

assert len(train_set) == 350
assert len(validation_set) == 75
assert len(test_set) == 75

assert train_set.isdisjoint(validation_set)
assert train_set.isdisjoint(test_set)
assert validation_set.isdisjoint(test_set)

patient_registry = pd.DataFrame(
    [
        {
            "public_id": pid,
            "split": "train",
            "source": next(
                source
                for source, ids
                in source_patient_sets.items()
                if pid in ids
            ),
        }
        for pid in train_patients
    ]
    + [
        {
            "public_id": pid,
            "split": "validation",
            "source": next(
                source
                for source, ids
                in source_patient_sets.items()
                if pid in ids
            ),
        }
        for pid in validation_patients
    ]
    + [
        {
            "public_id": pid,
            "split": "test",
            "source": next(
                source
                for source, ids
                in source_patient_sets.items()
                if pid in ids
            ),
        }
        for pid in test_patients
    ]
)

patient_registry.to_csv(
    PATCH_ROOT / "patient_split_500.csv",
    index=False,
)

print(
    patient_registry.groupby(
        ["source", "split"]
    ).size()
)

print()
print("Train:", len(train_patients))
print("Validation:", len(validation_patients))
print("Internal test:", len(test_patients))
print("Patient-level split integrity: PASS")

## Important split note

Notebook 02's final registry was already validated as:

- 500 unique patients
- 300 Part 1
- 120 Part 2
- 80 labelled validation
- 350 train
- 75 validation
- 75 internal test
- zero overlap

This notebook does not create a patch-level split. It reproduces the patient allocation before any patch is generated.

In [ ]:
# Cell 9 — Patient annotation maps

metadata_by_patient = {
    pid: group
    for pid, group
    in combined_metadata.groupby("public_id")
}


def get_label_map(patient_id):
    rows = metadata_by_patient.get(
        str(patient_id)
    )

    if rows is None:
        raise ValueError(
            f"No metadata found for {patient_id}"
        )

    return {
        int(row.label_id): int(row.label_code)
        for row in rows.itertuples()
    }


def get_source(patient_id):
    row = patient_registry.loc[
        patient_registry["public_id"]
        == str(patient_id)
    ]

    if row.empty:
        raise ValueError(
            f"{patient_id} not found in registry."
        )

    return row.iloc[0]["source"]


print("Patient annotation/source lookup ready.")

In [ ]:
# Cell 10 — NIfTI loading and validated preprocessing

def normalize_ct_hu(ct):
    ct = np.asarray(
        ct,
        dtype=np.float32,
    )

    ct = np.clip(
        ct,
        HU_MIN,
        HU_MAX,
    )

    ct = (
        ct - HU_MIN
    ) / (
        HU_MAX - HU_MIN
    )

    return ct.astype(
        np.float32,
        copy=False,
    )


def resample_volume(
    volume,
    original_spacing,
    target_spacing,
    order,
):
    original_spacing = np.asarray(
        original_spacing,
        dtype=np.float32,
    )

    target_spacing = np.asarray(
        target_spacing,
        dtype=np.float32,
    )

    zoom_factors = (
        original_spacing
        / target_spacing
    )

    return zoom(
        volume,
        zoom=zoom_factors,
        order=order,
        mode="nearest",
        prefilter=(order != 0),
    )


def load_patient_nifti_from_source(
    patient_id,
    source_name,
):
    cfg = SOURCES[source_name]

    index = get_source_zip_index(
        source_name
    )

    entries = index["entries"]

    image_member = find_patient_member(
        entries,
        patient_id,
        "image",
    )

    print(
        "Image member:",
        image_member,
    )

    image_bytes = get_zip_member_bytes(
        cfg["image_url"],
        entries[image_member],
    )

    label_zip_path = CACHE_ROOT / cfg["label_zip"]

    cache_small_file(
        cfg["label_url"],
        label_zip_path,
    )

    with zipfile.ZipFile(
        label_zip_path,
        "r",
    ) as zf:

        label_names = zf.namelist()

        candidates = [
            name
            for name in label_names
            if str(patient_id)
            in Path(name).name
            and "label"
            in Path(name).name.lower()
        ]

        if not candidates:
            raise FileNotFoundError(
                f"No label member found for "
                f"{patient_id} in "
                f"{cfg['label_zip']}"
            )

        label_member = sorted(
            candidates
        )[0]

        label_bytes = zf.read(
            label_member
        )

    print(
        "Label member:",
        label_member,
    )

    with tempfile.TemporaryDirectory() as tmpdir:

        image_path = (
            Path(tmpdir)
            / f"{patient_id}-image.nii.gz"
        )

        label_path = (
            Path(tmpdir)
            / f"{patient_id}-label.nii.gz"
        )

        image_path.write_bytes(
            image_bytes
        )

        label_path.write_bytes(
            label_bytes
        )

        ct_nii = nib.load(
            str(image_path)
        )

        label_nii = nib.load(
            str(label_path)
        )

        ct = ct_nii.get_fdata(
            dtype=np.float32
        )

        label = label_nii.get_fdata(
            dtype=np.float32
        )

        ct_spacing = np.asarray(
            ct_nii.header.get_zooms()[:3],
            dtype=np.float32,
        )

        label_spacing = np.asarray(
            label_nii.header.get_zooms()[:3],
            dtype=np.float32,
        )

        if not np.allclose(
            ct_spacing,
            label_spacing,
            atol=1e-4,
        ):
            raise ValueError(
                "CT/label spacing mismatch: "
                f"{ct_spacing} vs {label_spacing}"
            )

        if ct.shape != label.shape:
            raise ValueError(
                "CT/label shape mismatch: "
                f"{ct.shape} vs {label.shape}"
            )

    ct = normalize_ct_hu(ct)

    ct_resampled = resample_volume(
        ct,
        ct_spacing,
        TARGET_SPACING,
        order=1,
    ).astype(
        np.float32
    )

    label_resampled = resample_volume(
        label,
        label_spacing,
        TARGET_SPACING,
        order=0,
    ).astype(
        np.int16
    )

    label_resampled = np.rint(
        label_resampled
    ).astype(
        np.int16
    )

    if (
        ct_resampled.shape
        != label_resampled.shape
    ):
        raise ValueError(
            "Resampled CT/label mismatch: "
            f"{ct_resampled.shape} vs "
            f"{label_resampled.shape}"
        )

    return {
        "ct": ct_resampled,
        "label": label_resampled,
        "original_spacing": ct_spacing.tolist(),
        "target_spacing": TARGET_SPACING.tolist(),
        "patient_id": str(patient_id),
        "source": source_name,
    }


def preprocess_patient(
    patient_id,
):
    source_name = get_source(
        patient_id
    )

    return load_patient_nifti_from_source(
        patient_id,
        source_name,
    )


print("On-demand preprocessing ready.")

In [ ]:
# Cell 11 — Validate preprocessing across all three sources

validation_patients_by_source = {
    "train_part1": sorted(
        source_patient_sets["train_part1"]
    )[0],
    "train_part2": sorted(
        source_patient_sets["train_part2"]
    )[0],
    "validation": sorted(
        source_patient_sets["validation"]
    )[0],
}

cross_source_results = {}

for source_name, pid in validation_patients_by_source.items():

    print("\n" + "=" * 80)
    print(
        f"TESTING {source_name}: {pid}"
    )
    print("=" * 80)

    result = preprocess_patient(pid)

    assert result["ct"].ndim == 3
    assert result["label"].ndim == 3
    assert result["ct"].shape == result["label"].shape
    assert np.isfinite(result["ct"]).all()
    assert np.isfinite(result["label"]).all()
    assert 0.0 <= float(result["ct"].min())
    assert float(result["ct"].max()) <= 1.0

    cross_source_results[source_name] = {
        "patient": pid,
        "shape": result["ct"].shape,
        "spacing": result["original_spacing"],
        "label_ids": sorted(
            np.unique(
                result["label"]
            ).astype(int).tolist()
        ),
    }

    print(
        "Original spacing:",
        result["original_spacing"],
    )

    print(
        "Preprocessed shape:",
        result["ct"].shape,
    )

    print(
        "CT range:",
        float(result["ct"].min()),
        "to",
        float(result["ct"].max()),
    )

    print(
        "Label IDs:",
        cross_source_results[source_name]["label_ids"],
    )

    del result
    gc.collect()

print("\nCROSS-SOURCE PREPROCESSING: PASS")

In [ ]:
# Cell 12 — Build annotation masks

def build_masks(label, label_map):
    valid_mask = np.zeros(
        label.shape,
        dtype=np.uint8,
    )

    ambiguous_mask = np.zeros(
        label.shape,
        dtype=np.uint8,
    )

    unknown_mask = np.zeros(
        label.shape,
        dtype=np.uint8,
    )

    for label_id, label_code in label_map.items():

        label_id = int(label_id)
        label_code = int(label_code)

        if label_id == 0:
            continue

        if label_code in (
            1,
            2,
            3,
            4,
        ):
            valid_mask[
                label == label_id
            ] = 1

        elif label_code == AMBIGUOUS_LABEL_CODE:
            ambiguous_mask[
                label == label_id
            ] = 1

        else:
            unknown_mask[
                label == label_id
            ] = 1

    return (
        valid_mask,
        ambiguous_mask,
        unknown_mask,
    )


def classify_patch(
    label_patch,
    label_map,
):
    class_voxel_counts = {}

    for raw_id in np.unique(
        label_patch
    ):

        raw_id = int(raw_id)

        if raw_id == 0:
            continue

        label_code = label_map.get(
            raw_id
        )

        if label_code is None:
            return None, "unknown_annotation"

        if label_code == AMBIGUOUS_LABEL_CODE:
            return None, "ambiguous"

        if label_code not in LABEL_CODE_TO_CLASS:
            return None, "unsupported_label_code"

        class_id = LABEL_CODE_TO_CLASS[
            label_code
        ]

        class_voxel_counts[class_id] = (
            class_voxel_counts.get(
                class_id,
                0,
            )
            + int(
                np.sum(
                    label_patch == raw_id
                )
            )
        )

    if not class_voxel_counts:
        return 0, "background"

    maximum = max(
        class_voxel_counts.values()
    )

    dominant = [
        cls
        for cls, count
        in class_voxel_counts.items()
        if count == maximum
    ]

    if len(dominant) != 1:
        return None, "mixed_class_tie"

    return dominant[0], "positive"


print("Annotation mask/classification functions ready.")

In [ ]:
# Cell 13 — Patch coordinate generation

def generate_patch_starts(volume_shape):
    starts = []

    for z in range(
        0,
        volume_shape[0]
        - PATCH_SIZE[0]
        + 1,
        STRIDE[0],
    ):

        for y in range(
            0,
            volume_shape[1]
            - PATCH_SIZE[1]
            + 1,
            STRIDE[1],
        ):

            for x in range(
                0,
                volume_shape[2]
                - PATCH_SIZE[2]
                + 1,
                STRIDE[2],
            ):

                starts.append(
                    (z, y, x)
                )

    return starts


print(
    "Patch coordinate generator ready."
)

## Patient extraction

Each patient is processed independently.

For every candidate patch:

1. If it contains an ambiguous annotation → exclude.
2. If it contains an unsupported/unknown annotation ID → exclude.
3. If valid fracture classes are present but tied for dominance → exclude.
4. If no valid fracture is present → background candidate.
5. If a valid fracture is present with a unique dominant class → positive candidate.

The selected patch arrays are written using NumPy memory-mapped files so the notebook does not build a large in-RAM array for an entire patient.

In [ ]:
# Cell 14 — Extract one patient using disk-backed patch arrays

def stable_patient_seed(patient_id):
    digest = hashlib.sha256(
        str(patient_id).encode("utf-8")
    ).hexdigest()

    return SEED + int(
        digest[:8],
        16,
    )


def extract_patient(
    patient_id,
    split_name,
):
    print(
        f"Loading {patient_id} "
        f"({split_name})..."
    )

    result = preprocess_patient(
        patient_id
    )

    ct = result["ct"]
    label = result["label"]

    label_map = get_label_map(
        patient_id
    )

    (
        valid_mask,
        ambiguous_mask,
        unknown_mask,
    ) = build_masks(
        label,
        label_map,
    )

    starts = generate_patch_starts(
        ct.shape
    )

    positives = []
    negatives = []
    excluded = {
        "ambiguous": 0,
        "unknown_annotation": 0,
        "unsupported_label_code": 0,
        "mixed_class_tie": 0,
    }

    for start in starts:

        z, y, x = start

        z2 = z + PATCH_SIZE[0]
        y2 = y + PATCH_SIZE[1]
        x2 = x + PATCH_SIZE[2]

        label_patch = np.asarray(
            label[
                z:z2,
                y:y2,
                x:x2,
            ]
        )

        cls, reason = classify_patch(
            label_patch,
            label_map,
        )

        if reason in excluded:
            excluded[reason] += 1
            continue

        if np.any(
            unknown_mask[
                z:z2,
                y:y2,
                x:x2,
            ]
        ):
            excluded[
                "unknown_annotation"
            ] += 1
            continue

        if np.any(
            ambiguous_mask[
                z:z2,
                y:y2,
                x:x2,
            ]
        ):
            excluded["ambiguous"] += 1
            continue

        seg_patch = np.asarray(
            valid_mask[
                z:z2,
                y:y2,
                x:x2,
            ]
        )

        if cls == 0:

            if np.any(seg_patch):
                raise RuntimeError(
                    "Negative patch contains "
                    "valid fracture voxels."
                )

            negatives.append(
                (start, 0)
            )

        else:

            if not np.any(seg_patch):
                raise RuntimeError(
                    "Positive patch contains "
                    "no valid fracture voxels."
                )

            positives.append(
                (start, int(cls))
            )

    rng = random.Random(
        stable_patient_seed(
            patient_id
        )
    )

    rng.shuffle(positives)
    rng.shuffle(negatives)

    max_negative = int(
        len(positives)
        * NEGATIVE_TO_POSITIVE_RATIO
    )

    selected_negative = negatives[
        :max_negative
    ]

    selected = (
        positives
        + selected_negative
    )

    rng.shuffle(selected)

    out = (
        PATCH_ROOT
        / split_name
        / str(patient_id)
    )

    out.mkdir(
        parents=True,
        exist_ok=True,
    )

    n = len(selected)

    if n == 0:
        raise RuntimeError(
            f"{patient_id}: no usable patches "
            "were selected."
        )

    x_path = out / "X.npy"
    seg_path = out / "Y_seg.npy"
    class_path = out / "Y_class.npy"

    X = np.lib.format.open_memmap(
        x_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            n,
            *PATCH_SIZE,
            1,
        ),
    )

    Y_seg = np.lib.format.open_memmap(
        seg_path,
        mode="w+",
        dtype=np.uint8,
        shape=(
            n,
            *PATCH_SIZE,
            1,
        ),
    )

    Y_class = np.lib.format.open_memmap(
        class_path,
        mode="w+",
        dtype=np.int64,
        shape=(n,),
    )

    manifest_rows = []

    for i, (start, cls) in enumerate(
        selected
    ):

        z, y, x = start

        X[i, ..., 0] = np.asarray(
            ct[
                z:z + PATCH_SIZE[0],
                y:y + PATCH_SIZE[1],
                x:x + PATCH_SIZE[2],
            ],
            dtype=np.float32,
        )

        seg_patch = np.asarray(
            valid_mask[
                z:z + PATCH_SIZE[0],
                y:y + PATCH_SIZE[1],
                x:x + PATCH_SIZE[2],
            ],
            dtype=np.uint8,
        )

        Y_seg[i, ..., 0] = seg_patch
        Y_class[i] = int(cls)

        manifest_rows.append(
            {
                "public_id": str(patient_id),
                "split": split_name,
                "patch_index": i,
                "start_z": z,
                "start_y": y,
                "start_x": x,
                "classification": int(cls),
                "classification_name": CLASS_NAMES[
                    int(cls)
                ],
                "fracture_voxels": int(
                    seg_patch.sum()
                ),
            }
        )

    X.flush()
    Y_seg.flush()
    Y_class.flush()

    del X
    del Y_seg
    del Y_class

    pd.DataFrame(
        manifest_rows
    ).to_csv(
        out / "manifest.csv",
        index=False,
    )

    stats = {
        "public_id": str(patient_id),
        "split": split_name,
        "candidate_patches": len(starts),
        "positive_candidates": len(positives),
        "negative_candidates": len(negatives),
        "selected_positive": len(positives),
        "selected_negative": len(
            selected_negative
        ),
        "selected_total": len(selected),
        "excluded_ambiguous": excluded[
            "ambiguous"
        ],
        "excluded_unknown_annotation": excluded[
            "unknown_annotation"
        ],
        "excluded_unsupported_label_code": excluded[
            "unsupported_label_code"
        ],
        "excluded_mixed_class_tie": excluded[
            "mixed_class_tie"
        ],
    }

    del ct
    del label
    del valid_mask
    del ambiguous_mask
    del unknown_mask
    del result

    gc.collect()

    return stats


print("Patient extraction function ready.")

In [ ]:
# Cell 15 — Smoke test one patient from each source

SMOKE_TEST_PATIENTS = validation_patients_by_source

smoke_results = []

for source_name, patient_id in SMOKE_TEST_PATIENTS.items():

    print("\n" + "=" * 80)
    print(
        f"PATCH SMOKE TEST — {source_name} — "
        f"{patient_id}"
    )
    print("=" * 80)

    split_name = "train"

    stats = extract_patient(
        patient_id,
        split_name,
    )

    smoke_dir = (
        PATCH_ROOT
        / split_name
        / patient_id
    )

    X = np.load(
        smoke_dir / "X.npy",
        mmap_mode="r",
    )

    Y_seg = np.load(
        smoke_dir / "Y_seg.npy",
        mmap_mode="r",
    )

    Y_class = np.load(
        smoke_dir / "Y_class.npy",
        mmap_mode="r",
    )

    assert X.ndim == 5
    assert Y_seg.ndim == 5
    assert Y_class.ndim == 1

    assert X.shape[1:] == (
        *PATCH_SIZE,
        1,
    )

    assert Y_seg.shape[1:] == (
        *PATCH_SIZE,
        1,
    )

    assert (
        X.shape[0]
        == Y_seg.shape[0]
        == Y_class.shape[0]
    )

    assert set(
        np.unique(Y_seg)
    ).issubset({0, 1})

    assert set(
        np.unique(Y_class)
    ).issubset(
        set(CLASS_NAMES.keys())
    )

    smoke_results.append(
        {
            "source": source_name,
            "patient": patient_id,
            **stats,
            "X_shape": str(X.shape),
            "Y_seg_shape": str(
                Y_seg.shape
            ),
            "Y_class_shape": str(
                Y_class.shape
            ),
        }
    )

    print("Statistics:", stats)
    print("X shape:", X.shape)
    print("Y_seg shape:", Y_seg.shape)
    print("Y_class shape:", Y_class.shape)
    print("Smoke test: PASS")

    del X
    del Y_seg
    del Y_class
    gc.collect()

pd.DataFrame(
    smoke_results
).to_csv(
    PATCH_ROOT
    / "smoke_test_summary.csv",
    index=False,
)

print("\nThree-source patch smoke test: PASS")

In [ ]:
# Cell 16 — Full extraction control

RUN_FULL_EXTRACTION = False

print(
    "RUN_FULL_EXTRACTION =",
    RUN_FULL_EXTRACTION,
)

if not RUN_FULL_EXTRACTION:
    print(
        "Full extraction is disabled."
    )
    print(
        "After Cell 15 passes, set "
        "RUN_FULL_EXTRACTION = True "
        "and rerun this cell."
    )

In [ ]:
# Cell 17 — Process all 500 patients

split_patients = {
    "train": train_patients,
    "validation": validation_patients,
    "test": test_patients,
}

all_statistics = []

if RUN_FULL_EXTRACTION:

    for split_name, patients in split_patients.items():

        print("\n" + "=" * 80)
        print(
            f"PROCESSING {split_name.upper()} "
            f"({len(patients)} patients)"
        )
        print("=" * 80)

        for index, patient_id in enumerate(
            patients,
            start=1,
        ):

            print(
                f"\n[{index}/{len(patients)}] "
                f"{patient_id}"
            )

            try:

                stats = extract_patient(
                    patient_id,
                    split_name,
                )

                all_statistics.append(
                    stats
                )

                print(
                    "Completed:",
                    stats,
                )

            except Exception as exc:

                print(
                    f"ERROR processing "
                    f"{patient_id}: {exc}"
                )

                all_statistics.append(
                    {
                        "public_id": str(
                            patient_id
                        ),
                        "split": split_name,
                        "error": str(exc),
                    }
                )

            gc.collect()

    extraction_df = pd.DataFrame(
        all_statistics
    )

    extraction_df.to_csv(
        PATCH_ROOT
        / "full_extraction_summary.csv",
        index=False,
    )

    print(
        "\nFull extraction summary saved."
    )

else:
    print(
        "Full extraction not executed."
    )

In [ ]:
# Cell 18 — Final patch-dataset validation

def validate_patch_dataset():

    errors = []
    patient_rows = []

    for split_name, patients in split_patients.items():

        split_dir = (
            PATCH_ROOT
            / split_name
        )

        if not split_dir.exists():
            errors.append(
                f"Missing split directory: "
                f"{split_name}"
            )
            continue

        found = {
            p.name
            for p in split_dir.iterdir()
            if p.is_dir()
        }

        expected = set(
            patients
        )

        if found != expected:
            missing = expected - found
            extra = found - expected

            errors.append(
                f"{split_name}: "
                f"missing={len(missing)}, "
                f"extra={len(extra)}"
            )

        for patient_id in patients:

            patient_dir = (
                split_dir
                / patient_id
            )

            required_files = [
                "X.npy",
                "Y_seg.npy",
                "Y_class.npy",
                "manifest.csv",
            ]

            for filename in required_files:

                path = (
                    patient_dir
                    / filename
                )

                if not path.exists():
                    errors.append(
                        f"Missing: {path}"
                    )

            if not patient_dir.exists():
                continue

            try:

                X = np.load(
                    patient_dir / "X.npy",
                    mmap_mode="r",
                )

                Y_seg = np.load(
                    patient_dir / "Y_seg.npy",
                    mmap_mode="r",
                )

                Y_class = np.load(
                    patient_dir / "Y_class.npy",
                    mmap_mode="r",
                )

                if (
                    X.shape[0]
                    != Y_seg.shape[0]
                    or X.shape[0]
                    != Y_class.shape[0]
                ):
                    errors.append(
                        f"{patient_id}: "
                        "patch-count mismatch"
                    )

                if X.shape[1:] != (
                    *PATCH_SIZE,
                    1,
                ):
                    errors.append(
                        f"{patient_id}: "
                        f"invalid X shape {X.shape}"
                    )

                if Y_seg.shape[1:] != (
                    *PATCH_SIZE,
                    1,
                ):
                    errors.append(
                        f"{patient_id}: "
                        f"invalid Y_seg shape "
                        f"{Y_seg.shape}"
                    )

                if not set(
                    np.unique(Y_seg)
                ).issubset({0, 1}):

                    errors.append(
                        f"{patient_id}: "
                        "invalid segmentation values"
                    )

                if not set(
                    np.unique(Y_class)
                ).issubset(
                    set(CLASS_NAMES.keys())
                ):

                    errors.append(
                        f"{patient_id}: "
                        "invalid classification values"
                    )

                manifest = pd.read_csv(
                    patient_dir
                    / "manifest.csv"
                )

                if len(manifest) != X.shape[0]:
                    errors.append(
                        f"{patient_id}: "
                        "manifest/patch count mismatch"
                    )

                patient_rows.append(
                    {
                        "public_id": patient_id,
                        "split": split_name,
                        "patches": X.shape[0],
                    }
                )

                del X
                del Y_seg
                del Y_class

            except Exception as exc:

                errors.append(
                    f"{patient_id}: "
                    f"{exc}"
                )

    if errors:

        print(
            "PATCH DATASET VALIDATION: FAIL"
        )

        for error in errors[:100]:
            print(" -", error)

        raise RuntimeError(
            f"{len(errors)} validation errors."
        )

    validation_summary = pd.DataFrame(
        patient_rows
    )

    validation_summary.to_csv(
        PATCH_ROOT
        / "patch_dataset_patient_summary.csv",
        index=False,
    )

    print(
        "PATCH DATASET VALIDATION: PASS"
    )

    print(
        "Patients validated:",
        len(validation_summary),
    )

    print(
        "Total patches:",
        int(
            validation_summary[
                "patches"
            ].sum()
        ),
    )

    print(
        "\nPatches by split:"
    )

    print(
        validation_summary
        .groupby("split")["patches"]
        .agg(
            ["count", "sum", "mean"]
        )
    )


if RUN_FULL_EXTRACTION:
    validate_patch_dataset()
else:
    print(
        "Full patch-dataset validation "
        "skipped because extraction is disabled."
    )

In [ ]:
# Cell 19 — Save final extraction configuration

config = {
    "seed": SEED,
    "patch_size": list(PATCH_SIZE),
    "stride": list(STRIDE),
    "negative_to_positive_ratio": NEGATIVE_TO_POSITIVE_RATIO,
    "minimum_fracture_voxels": MIN_FRACTURE_VOXELS,
    "ambiguous_label_code": AMBIGUOUS_LABEL_CODE,
    "classification_classes": CLASS_NAMES,
    "label_code_to_class": LABEL_CODE_TO_CLASS,
    "target_spacing": TARGET_SPACING.tolist(),
    "hu_min": HU_MIN,
    "hu_max": HU_MAX,
    "patient_counts": {
        "total": 500,
        "train": 350,
        "validation": 75,
        "test": 75,
    },
    "source_counts": {
        "train_part1": 300,
        "train_part2": 120,
        "validation": 80,
    },
    "official_test_ground_truth_available": False,
    "full_extraction_executed": bool(
        RUN_FULL_EXTRACTION
    ),
}

with open(
    PATCH_ROOT
    / "patch_extraction_config.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        config,
        f,
        indent=2,
    )

print(
    "Configuration saved:",
    PATCH_ROOT
    / "patch_extraction_config.json",
)

# Notebook 03 completion criteria

Notebook 03 is complete when:

1. The 500 labelled patients are verified.
2. The 350/75/75 patient split is verified.
3. All three RibFrac sources pass preprocessing.
4. Annotation IDs are mapped using patient metadata.
5. Ambiguous (`-1`) annotations are excluded.
6. Patch size is `(64,64,64)`.
7. Segmentation targets contain only `{0,1}`.
8. Classification targets contain only `{0,1,2,3,4}`.
9. Positive patches contain valid fracture voxels.
10. Negative patches contain no valid fracture voxels.
11. Mixed-class ties are excluded.
12. CT, segmentation and classification patch counts agree.
13. Patch files reload successfully.
14. Full extraction is performed patient-by-patient.
15. No patch-level random split is performed.
16. The final patch dataset passes Cell 18 validation.

The resulting patch dataset can then be published as a Kaggle Dataset for Notebook 05 training.